# 01 — Data Ingestion & Cleaning
### DiscoverAI · Deloitte × LUISS

This notebook takes the two raw JSON files from McAuley-Lab/Amazon-Reviews-2023
and produces two clean, embedding-ready CSV files.

**Key design decisions (all data-driven):**
| Decision | Why |
|---|---|
| Min 3 verified reviews per product | Fewer reviews = noise, not signal |
| Balanced rating selection | Raw data is 61% 5-star — we fix this to 48% |
| Min 25 tokens per review | 48.7% of raw reviews are < 20 tokens ("Love it!!") |
| FORM field at top of product text | Disambiguates shampoo vs capsule vs cream |
| 18 semantic fields from `details` | 98% coverage — more useful than `description` |
| Cap 400 tokens per product | MPNet hard limit is 512 subword tokens |

**Output files:**
- `products_cleaned.csv` — one row per product, enriched `product_text_base`
- `reviews_cleaned.csv` — balanced, quality-filtered reviews


## 0 · Imports

In [ ]:
import json, re
import numpy as np
import pandas as pd
from html import unescape
from collections import defaultdict

DATA_DIR  = "/Users/danielegiovanardi/MSTERRORS/Mean-Squared-Terrors/data"
META_PATH = f"{DATA_DIR}/meta_Health_and_Personal_Care.json"
REV_PATH  = f"{DATA_DIR}/Health_and_Personal_Care.json"

print("Pronto.")


## 1 · Configuration

All parameters in one place. Change only here, never inside functions.


In [ ]:
# ── Prodotti ──────────────────────────────────────────────────────────────────
MIN_VERIFIED_REVIEWS = 3     # prodotti con meno review verified vengono esclusi
CAP_PRODUCT_TOKENS   = 400   # limite token testo prodotto (MPNet max ≈ 512 subword)
CAP_FEATURES_TOKENS  = 150   # max token da features
CAP_DESC_TOKENS      = 150   # max token da description
CAP_DETAILS_TOKENS   = 80    # max token da details semantici

# Chiavi details con valore semantico per le query (esclude dimensioni fisiche, date, codici)
SEMANTIC_DETAIL_KEYS = {
    'Item Form', 'Material', 'Material Feature', 'Special Feature',
    'Flavor', 'Dosage Form', 'Color', 'Size', 'Scent', 'Skin Type',
    'Active Ingredients', 'Specific Uses For Product', 'Product Benefits',
    'Compatible Devices', 'Power Source', 'Age Range (Description)',
    'Style', 'Recommended Uses For Product', 'Department', 'Occasion',
}

# ── Review ────────────────────────────────────────────────────────────────────
MIN_REVIEW_TOKENS = 25    # soglia minima: il 48.7% delle review ha < 20 tok

# Quante review per rating (selezione bilanciata per prodotto)
# Obiettivo: ~35% 5-star invece del 61% raw
REVIEWS_PER_RATING = {
    5: 5,   # max 5 review 5-star (le più helpful)
    4: 3,   # max 3 review 4-star
    3: 2,   # max 2 review 3-star (neutrali, poco segnale — cap basso)
    2: 3,   # max 3 review 2-star
    1: 3,   # max 3 review 1-star (critiche, segnale importante)
}
# Cap totale per prodotto = somma dei cap = 16, ma nella pratica sarà ~10-12
# perché non tutti i prodotti hanno review in tutte le fasce

print("Parametri:")
print(f"  Prodotti: min {MIN_VERIFIED_REVIEWS} review verified, cap testo {CAP_PRODUCT_TOKENS} token")
print(f"  Review: min {MIN_REVIEW_TOKENS} token, selezione bilanciata {REVIEWS_PER_RATING}")
print(f"  Cap totale review/prodotto: {sum(REVIEWS_PER_RATING.values())}")


## 2 · Text cleaning functions

### `clean_text(text)`
Strips HTML, normalises apostrophes and unicode dashes, collapses whitespace.

### `build_product_text(row_dict)`
Assembles `product_text_base` in the optimal order for embedding:
```
TITLE → BRAND → FORM → FEATURES → DESCRIPTION → SPECS
```
FORM (Item Form / Dosage Form) comes right after title so the model immediately
knows whether the product is a shampoo, a capsule, a cream, or a device — before
reading anything else. This was added in v4 after testing showed that without it
the model confused biotin shampoos with biotin supplements.

### `extract_details_semantic(details)`
Selects only the 18 semantically useful keys from `details` (Skin Type, Material,
Flavor, etc.) and discards metadata-only fields (Package Dimensions, Date First
Available, Item model number).


In [ ]:
def clean_text(text):
    """Strip HTML tags, normalise unicode chars, collapse whitespace. Returns None if empty."""
    if not text or (isinstance(text, float) and np.isnan(text)):
        return None
    text = unescape(str(text))
    text = text.replace('\u2019', "'").replace('\u2018', "'")
    text = text.replace('\u2013', '-').replace('\u2014', '-')
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^\w\s.,!?\'/\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if len(text) > 2 else None

def flatten_to_text(val, cap_tokens=None):
    """Convert list or string to clean text, with optional token cap."""
    if isinstance(val, list):
        parts = [clean_text(str(x)) for x in val if x and str(x).strip()]
        parts = [p for p in parts if p]
        text  = ' '.join(parts)
    elif isinstance(val, str):
        text = clean_text(val)
    else:
        return None
    if not text:
        return None
    if cap_tokens:
        words = text.split()
        text  = ' '.join(words[:cap_tokens])
    return text if text else None

# Campi details semantici — espanso con tutti i campi di forma e uso
SEMANTIC_DETAIL_KEYS = {
    'Item Form', 'Material', 'Material Feature', 'Special Feature',
    'Flavor', 'Dosage Form', 'Color', 'Size', 'Scent', 'Skin Type',
    'Active Ingredients', 'Specific Uses For Product', 'Product Benefits',
    'Compatible Devices', 'Power Source', 'Age Range (Description)',
    'Style', 'Recommended Uses For Product', 'Department', 'Occasion',
    'Target Audience', 'Hair Type', 'Skin Tone',
}

# Campi di forma del prodotto — vanno in cima per disambiguare categoria
FORM_KEYS = {'Item Form', 'Dosage Form', 'Material'}

def extract_details_semantic(details, cap_tokens=None):
    """Estrae i campi semantici da details."""
    if not isinstance(details, dict):
        return None, None   # (form_text, rest_text)

    form_parts = []
    rest_parts = []

    for k, v in details.items():
        if k not in SEMANTIC_DETAIL_KEYS:
            continue
        v_str = str(v).strip()
        if not v_str or v_str.lower() in ('none', 'n/a', '-', '—', ''):
            continue
        if k in FORM_KEYS:
            form_parts.append(f"{k}: {v_str}")
        else:
            rest_parts.append(f"{k}: {v_str}")

    form_text = '. '.join(form_parts) if form_parts else None
    rest_text = '. '.join(rest_parts) if rest_parts else None

    if cap_tokens and rest_text:
        rest_text = ' '.join(rest_text.split()[:cap_tokens])

    return form_text, rest_text

def get_brand(row_dict):
    """Extract brand from multiple fallback sources."""
    for field in ('brand', 'store'):
        val = row_dict.get(field)
        if val and str(val).strip().lower() not in ('none', ''):
            return clean_text(str(val))
    d = row_dict.get('details')
    if isinstance(d, dict):
        for key in ('Brand', 'Manufacturer'):
            val = d.get(key)
            if val and str(val).strip().lower() not in ('none', ''):
                return clean_text(str(val))
    return None

def build_product_text(row_dict):
    """
    Costruisce product_text_base con Item Form in cima.

    Ordine: TITLE → BRAND → FORM (Item Form/Dosage Form) → FEATURES → DESCRIPTION → SPECS
    Item Form in cima disambigua subito la categoria:
    'FORM: Shampoo' vs 'FORM: Capsule' vs 'FORM: Cream'
    """
    parts = []

    # 1. Titolo
    title = clean_text(row_dict.get('title'))
    if title:
        parts.append(f"TITLE: {title}")

    # 2. Brand
    brand = get_brand(row_dict)
    if brand:
        parts.append(f"BRAND: {brand}")

    # 3. Form — NUOVO: in cima per disambiguare categoria del prodotto
    form_text, details_rest = extract_details_semantic(
        row_dict.get('details'), cap_tokens=CAP_DETAILS_TOKENS
    )
    if form_text:
        parts.append(f"FORM: {form_text}")

    # 4. Features (bullet points)
    features = flatten_to_text(row_dict.get('features'), cap_tokens=CAP_FEATURES_TOKENS)
    if features:
        parts.append(f"FEATURES: {features}")

    # 5. Description
    desc = flatten_to_text(row_dict.get('description'), cap_tokens=CAP_DESC_TOKENS)
    if desc:
        parts.append(f"DESCRIPTION: {desc}")

    # 6. Resto dei details semantici
    if details_rest:
        parts.append(f"SPECS: {details_rest}")

    full_text = '\n'.join(parts)
    words     = full_text.split()
    if len(words) > CAP_PRODUCT_TOKENS:
        full_text = ' '.join(words[:CAP_PRODUCT_TOKENS])

    return full_text

print("Functions ready (v4 — with FORM disambiguation).")

# Sanity check: verify that FORM field disambiguates product type
test_shampoo = {
    'title': 'Biotin Shampoo for Hair Growth Anti Hair Loss',
    'brand': 'Pura D or',
    'features': ['Stimulates hair growth', 'Contains biotin and saw palmetto'],
    'description': ['Professional hair thickening shampoo'],
    'details': {
        'Item Form': 'Shampoo',
        'Scent': 'Fresh',
        'Hair Type': 'Thinning',
        'Active Ingredients': 'Biotin, Saw Palmetto',
    }
}
test_supplement = {
    'title': 'Biotin 10000mcg Hair Growth Supplement',
    'brand': 'Nature Bounty',
    'features': ['Supports hair, skin and nail growth'],
    'description': ['High potency biotin supplement'],
    'details': {
        'Item Form': 'Capsule',
        'Dosage Form': 'Capsule',
        'Active Ingredients': 'Biotin 10000mcg',
    }
}

print("\n--- Shampoo product ---")
print(build_product_text(test_shampoo)[:300])
print("\n--- Supplement product ---")
print(build_product_text(test_supplement)[:300])
print("\n-> The model now sees 'FORM: Shampoo' vs 'FORM: Capsule' right after the title")


## 3 · Load and process product metadata

Reads all 60,293 products from the raw JSON. For each product, calls
`build_product_text()` to assemble the enriched `product_text_base`.


In [ ]:
print("Caricamento metadata raw...")
meta_raw = []
with open(META_PATH) as f:
    for line in f:
        try: meta_raw.append(json.loads(line))
        except: pass
print(f"Prodotti raw: {len(meta_raw):,}")


In [ ]:
print("Building enriched product_text_base for all products...")
products = []
for row in meta_raw:
    asin = row.get('parent_asin')
    if not asin:
        continue

    text = build_product_text(row)
    tok_len = len(text.split()) if text else 0

    products.append({
        'parent_asin':          asin,
        'product_title':        clean_text(row.get('title')),
        'brand':                get_brand(row),
        'description_clean':    flatten_to_text(row.get('description'), cap_tokens=CAP_DESC_TOKENS),
        'features_clean':       flatten_to_text(row.get('features'), cap_tokens=CAP_FEATURES_TOKENS),
        'details_text':         extract_details_semantic(row.get('details')),
        'product_avg_rating':   row.get('average_rating'),
        'product_rating_count': row.get('rating_number'),
        'price':                row.get('price'),
        'store':                row.get('store'),
        'product_text_base':    text,
        'text_tokens':          tok_len,
    })

df_meta = pd.DataFrame(products).drop_duplicates('parent_asin').reset_index(drop=True)
print(f"Prodotti unici: {len(df_meta):,}")

# Statistiche copertura
print("\nCopertura campi:")
for col in ['product_title','brand','features_clean','description_clean','details_text']:
    print(f"  {col:25s}: {df_meta[col].notna().mean():.1%}")

print(f"\nToken count per product (product_text_base):")
print(f"  mean={df_meta['text_tokens'].mean():.0f}  "
      f"median={df_meta['text_tokens'].median():.0f}  "
      f"min={df_meta['text_tokens'].min()}  max={df_meta['text_tokens'].max()}")
print(f"  < 15 tokens (poor text): {(df_meta['text_tokens'] < 15).mean():.1%}")
print(f"  > 50 tokens (rich text):  {(df_meta['text_tokens'] > 50).mean():.1%}")
print(f"  > 100 token:               {(df_meta['text_tokens'] > 100).mean():.1%}")


## 4 · Load, filter and select reviews

**Filtering pipeline:**
1. `verified_purchase == True` — Amazon's quality signal
2. `tok_len >= 25` — removes noise ("Love it!!", "Fast shipping", etc.)
3. Only products present in metadata

**Balanced rating selection:**
Instead of top-10 by helpful_vote (which gives 70% 5-star reviews), we pick:
- max 5 × 5-star (most helpful first)
- max 3 × 4-star
- max 2 × 3-star
- max 3 × 2-star
- max 3 × 1-star

This brings the 5-star bias from 70% down to 48%.


In [ ]:
print("Loading raw reviews...")
rev_raw = []
with open(REV_PATH) as f:
    for line in f:
        try: rev_raw.append(json.loads(line))
        except: pass
print(f"Raw reviews: {len(rev_raw):,}")

df_rev = pd.DataFrame(rev_raw)
df_rev['text']  = df_rev['text'].apply(clean_text)
df_rev['title'] = df_rev['title'].apply(lambda x: clean_text(x) if pd.notna(x) else None)
df_rev['helpful_vote'] = pd.to_numeric(df_rev['helpful_vote'], errors='coerce').fillna(0).astype(int)
df_rev['tok_len'] = df_rev['text'].fillna('').str.split().str.len()

print(f"Reviews after loading: {len(df_rev):,}")


In [ ]:
# Filter 1: verified purchases only ───────────────────────────────────────────────
df_rev = df_rev[df_rev['verified_purchase'] == True].copy()
print(f"After verified filter:      {len(df_rev):,}")

# Filter 2: minimum token threshold ────────────────────────────────────────────────────
df_rev = df_rev[df_rev['tok_len'] >= MIN_REVIEW_TOKENS].copy()
print(f"After tok >= {MIN_REVIEW_TOKENS}:     {len(df_rev):,}")

# Filter 3: only products present in metadata ─────────────────────────────────────
valid_asins = set(df_meta['parent_asin'])
df_rev = df_rev[df_rev['parent_asin'].isin(valid_asins)].copy()
print(f"After valid asin filter:   {len(df_rev):,}")
print(f"Prodotti con ≥1 review: {df_rev['parent_asin'].nunique():,}")


In [ ]:
# Balanced selection by rating — reduces 5-star bias from 70% to 48% ───────────────────────────────────────────
print("Balanced rating selection...")

def select_balanced(grp):
    """Select reviews with balanced rating distribution per product."""
    """
    Per ogni prodotto seleziona le review in modo bilanciato per rating.
    A parità di rating, prende quelle con più helpful_vote.
    """
    selected = []
    for rating, cap in REVIEWS_PER_RATING.items():
        bucket = grp[grp['rating'] == rating].nlargest(cap, 'helpful_vote')
        selected.append(bucket)
    return pd.concat(selected) if selected else grp.head(0)

df_selected = (
    df_rev
    .groupby('parent_asin', group_keys=False)
    .apply(select_balanced)
    .reset_index(drop=True)
)

# ── Filtro 4: solo prodotti con ≥ MIN_VERIFIED_REVIEWS review selezionate ──────
rpp = df_selected.groupby('parent_asin').size()
valid_prods = rpp[rpp >= MIN_VERIFIED_REVIEWS].index
df_selected = df_selected[df_selected['parent_asin'].isin(valid_prods)].copy()

print(f"\nReview selezionate:    {len(df_selected):,}")
print(f"Products covered:      {df_selected['parent_asin'].nunique():,}")
rpp_final = df_selected.groupby('parent_asin').size()
print(f"Reviews/product: mean={rpp_final.mean():.1f}  "
      f"mediana={rpp_final.median():.0f}  max={rpp_final.max()}")


In [ ]:
# Rating distribution after balancing
print("Rating distribution (target: reduce 5-star bias):")
total = len(df_selected)
for r in [1,2,3,4,5]:
    n   = (df_selected['rating'] == r).sum()
    bar = '█' * int(n/total*50)
    print(f"  {r}⭐ {bar:<50} {n:,} ({n/total:.1%})")

print(f"\nReview token stats:")
print(f"  media={df_selected['tok_len'].mean():.0f}  "
      f"mediana={df_selected['tok_len'].median():.0f}  "
      f"p90={df_selected['tok_len'].quantile(0.9):.0f}")
print(f"  With helpful_vote > 0: {(df_selected['helpful_vote'] > 0).mean():.1%}")


## 5 · Align the two datasets

In [ ]:
# Keep only products that have enough reviews
final_asins = set(df_selected['parent_asin'])
df_products = df_meta[df_meta['parent_asin'].isin(final_asins)].copy().reset_index(drop=True)

print(f"Final products: {len(df_products):,}")
print(f"Final reviews:   {len(df_selected):,}")
print(f"\nSanity checks:")
print(f"  All products in reviews exist in metadata: "
      f"{final_asins.issubset(set(df_products['parent_asin']))}")
print(f"  Ogni prodotto ha >= {MIN_VERIFIED_REVIEWS} review: "
      f"{df_selected.groupby('parent_asin').size().min() >= MIN_VERIFIED_REVIEWS}")
print(f"  No reviews with empty text: "
      f"{(df_selected['text'].isna() | (df_selected['text'] == '')).sum() == 0}")


## 6 · Text quality analysis

Three charts:
1. Field coverage (brand, features, description, details, price)
2. Token distribution per product — before and after v1→v3
3. Rating distribution in reviews after balancing


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Qualità dataset prodotti v3", fontweight='bold')

# Distribuzione token
sns.histplot(df_products['text_tokens'].clip(upper=400), bins=50,
             ax=axes[0], color='steelblue', kde=True)
axes[0].axvline(df_products['text_tokens'].median(), color='red',
                ls='--', lw=1.5, label=f"mediana={df_products['text_tokens'].median():.0f}")
axes[0].set(title='Token per prodotto', xlabel='Token', ylabel='# Prodotti')
axes[0].legend()

# Copertura campi
fields = ['product_title','brand','features_clean','description_clean','details_text']
labels = ['Titolo','Brand','Features','Description','Details']
pcts   = [df_products[f].notna().mean()*100 for f in fields]
colors = ['#1D9E75' if p>60 else '#BA7517' if p>30 else '#D85A30' for p in pcts]
axes[1].barh(labels, pcts, color=colors)
axes[1].axvline(100, color='grey', ls='--', lw=0.8)
axes[1].set(title='Copertura campi (%)', xlabel='%')
for i, p in enumerate(pcts):
    axes[1].text(p+1, i, f'{p:.0f}%', va='center', fontsize=10)

# Distribuzione rating nelle review
counts = [df_selected[df_selected['rating']==r].shape[0] for r in [1,2,3,4,5]]
colors_r = ['#E24B4A','#F09595','#FAC775','#97C459','#1D9E75']
axes[2].bar([1,2,3,4,5], counts, color=colors_r)
axes[2].set(title='Rating distribution (review v3)', xlabel='Stelle', ylabel='# Review')
total_r = sum(counts)
for i, (r, c) in enumerate(zip([1,2,3,4,5], counts)):
    axes[2].text(r, c+200, f'{c/total_r:.0%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"Remaining poor-text products (<15 tok): {(df_products['text_tokens']<15).mean():.1%}")
print(f"Rich-text products (>50 tok):          {(df_products['text_tokens']>50).mean():.1%}")


## 7 · Save datasets

Saves `products_cleaned.csv` and `reviews_cleaned.csv` to DATA_DIR.
Verifies the saved files by reading them back and checking row counts.


In [ ]:
import os

# products_cleaned.csv — only products with enough reviews ─────────────
final_asins = set(df_selected['parent_asin'])   # products with >= 3 reviews

product_cols = [
    'parent_asin', 'product_title', 'brand', 'description_clean',
    'features_clean', 'details_text', 'product_avg_rating',
    'product_rating_count', 'price', 'store', 'product_text_base', 'text_tokens'
]

# Filtra df_meta (non df_products che potrebbe non essere aggiornato)
products_to_save = df_meta[df_meta['parent_asin'].isin(final_asins)][product_cols].copy()
products_to_save = products_to_save.drop_duplicates('parent_asin').reset_index(drop=True)
products_to_save.to_csv(f"{DATA_DIR}/products_cleaned.csv", index=False)

# ── reviews_cleaned.csv ───────────────────────────────────────────────────────
review_cols = ['parent_asin', 'title', 'text', 'rating',
               'verified_purchase', 'helpful_vote', 'timestamp', 'tok_len']
review_cols = [c for c in review_cols if c in df_selected.columns]
df_selected[review_cols].to_csv(f"{DATA_DIR}/reviews_cleaned.csv", index=False)

# Log
print("Files saved:")
for fname in ["products_cleaned.csv", "reviews_cleaned.csv"]:
    path  = f"{DATA_DIR}/{fname}"
    size  = os.path.getsize(path) / 1e6
    lines = sum(1 for _ in open(path)) - 1
    print(f"  {fname:<30} {size:6.1f} MB   {lines:,} righe")

print(f"\n{'='*50}")
print(f"  RIEPILOGO FINALE v3")
print(f"{'='*50}")
print(f"  Prodotti            : {len(products_to_save):,}  ← deve essere ~11,537")
print(f"  Review              : {len(df_selected):,}")
print(f"  Review/prodotto     : {len(df_selected)/len(products_to_save):.1f} media")
print(f"  Testo medio prodotto: {products_to_save['text_tokens'].mean():.0f} token")
print(f"  Testo povero (<15t) : {(products_to_save['text_tokens']<15).mean():.1%}")
print(f"  5-star bias         : {(df_selected['rating']==5).mean():.1%}")
print(f"  Token medi review   : {df_selected['tok_len'].mean():.0f}")


## 8 · Sample outputs — verify the result

In [ ]:
# Show 3 representative products
print("=== Product examples (rich, medium, poor text) ===\n")
for label, cond in [
    ("RICH (>100 tok)",  df_products['text_tokens'] > 100),
    ("MEDIUM (30-60 tok)", (df_products['text_tokens'] >= 30) & (df_products['text_tokens'] <= 60)),
    ("POOR (<20 tok)",  df_products['text_tokens'] < 20),
]:
    subset = df_products[cond]
    if len(subset) == 0: continue
    ex = subset.iloc[0]
    print(f"--- {label} ---")
    print(ex['product_text_base'][:350])
    print(f"[{ex['text_tokens']} token]\n")

# Mostra review per un prodotto con distribuzione bilanciata
print("\n=== Example of balanced reviews for one product ===")
ex_asin = df_products.iloc[10]['parent_asin']
ex_revs = df_selected[df_selected['parent_asin'] == ex_asin].sort_values('rating')
print(f"Product: {df_products.iloc[10]['product_title'][:60]}")
print(f"Selected reviews: {len(ex_revs)}")
for _, r in ex_revs.iterrows():
    print(f"  {int(r['rating'])}⭐ hv={int(r['helpful_vote'])} tok={int(r['tok_len'])}  {str(r['text'])[:80]}...")
